<a href="https://colab.research.google.com/github/njshrutics24-bot/AI/blob/main/1BF24CS185_Week9_FOL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
class Term:
    def __init__(self, name, is_var=False):
        self.name = name
        self.is_var = is_var

    def __repr__(self):
        return self.name

    def __eq__(self, other):
        return isinstance(other, Term) and self.name == other.name and self.is_var == other.is_var

    def __hash__(self):
        return hash((self.name, self.is_var))


class Predicate:
    def __init__(self, name, args, negated=False):
        self.name = name
        self.args = args
        self.negated = negated

    def __repr__(self):
        sign = "~" if self.negated else ""
        return f"{sign}{self.name}({', '.join(map(str, self.args))})"

    def __eq__(self, other):
        return (
            isinstance(other, Predicate)
            and self.name == other.name
            and self.args == other.args
            and self.negated == other.negated
        )

    def __hash__(self):
        return hash((self.name, tuple(self.args), self.negated))


def occurs_check(var, x, theta):
    if var == x:
        return True
    if x in theta:
        return occurs_check(var, theta[x], theta)
    return False


def unify_var(var, x, theta):
    if var in theta:
        return unify(theta[var], x, theta)
    if x in theta:
        return unify(var, theta[x], theta)
    if occurs_check(var, x, theta):
        return None
    theta[var] = x
    return theta


def unify(x, y, theta=None):
    if theta is None:
        theta = {}
    if x == y:
        return theta
    if isinstance(x, Term) and x.is_var:
        return unify_var(x, y, theta)
    if isinstance(y, Term) and y.is_var:
        return unify_var(y, x, theta)
    if isinstance(x, list) and isinstance(y, list) and len(x) == len(y):
        for xi, yi in zip(x, y):
            theta = unify(xi, yi, theta)
            if theta is None:
                return None
        return theta
    return None


def substitute(clause, theta):
    new_lits = []
    for lit in clause:
        new_args = []
        for arg in lit.args:
            while arg in theta:
                arg = theta[arg]
            new_args.append(arg)
        new_lits.append(Predicate(lit.name, new_args, lit.negated))
    return tuple(new_lits)


def resolve_pair(c1, c2):
    for l1 in c1:
        for l2 in c2:
            if l1.name == l2.name and l1.negated != l2.negated:
                theta = unify(l1.args, l2.args)
                if theta is not None:
                    rem1 = [l for l in c1 if l != l1]
                    rem2 = [l for l in c2 if l != l2]
                    res = substitute(rem1 + rem2, theta)
                    return res, theta
    return None, None


def format_clause(c):
    if not c:
        return "{   } Hence proved."
    return " V ".join(map(str, c))


def format_theta(theta):
    if not theta:
        return ""
    pairs = [f"{val}/{var}" for var, val in theta.items()]
    return "{" + ", ".join(pairs) + "}"


def print_resolution_tree(kb, negated_goal):
    current = negated_goal
    used = set()
    indent = 0

    while current:
        for idx, clause in enumerate(kb):
            if idx in used:
                continue
            resolvent, theta = resolve_pair(current, clause)
            if resolvent is not None:
                used.add(idx)
                left_str = format_clause(current)
                right_str = format_clause(clause)
                sub_str = format_theta(theta)
                pad = " " * indent

                print(f"{pad}{left_str:<28} {right_str}")
                print(f"{pad}   \\                       /")
                if sub_str:
                    print(f"{pad}    \\                     /    {sub_str}")
                else:
                    print(f"{pad}    \\                     /")
                print(f"{pad}     \\                   /")

                indent += 5
                current = resolvent
                break

    print(f"{' ' * indent}{format_clause(current)}")


# Terms
x = Term("x", is_var=True)
y = Term("y", is_var=True)
z = Term("z", is_var=True)
k = Term("k", is_var=True)

John = Term("John")
Peanuts = Term("Peanuts")
Apple = Term("Apple")
Vegetables = Term("Vegetables")
Anil = Term("Anil")
Harry = Term("Harry")

# Standardized CNF Clauses
kb_clauses = [
    (Predicate("Food", [x], negated=True), Predicate("Likes", [John, x])),
    (Predicate("Food", [Apple]),),
    (Predicate("Food", [Vegetables]),),
    (Predicate("Eats", [y, z], negated=True), Predicate("Killed", [y]), Predicate("Food", [z])),
    (Predicate("Eats", [Anil, Peanuts]),),
    (Predicate("Alive", [Anil]),),
    (Predicate("Eats", [Anil, x], negated=True), Predicate("Eats", [Harry, x])),
    (Predicate("Alive", [k], negated=True), Predicate("Killed", [k], negated=True)),
    (Predicate("Killed", [k]), Predicate("Alive", [k])),
]

negated_goal = (Predicate("Likes", [John, Peanuts], negated=True),)

print_resolution_tree(kb_clauses, negated_goal)


~Likes(John, Peanuts)        ~Food(x) V Likes(John, x)
   \                       /
    \                     /    {Peanuts/x}
     \                   /
     ~Food(Peanuts)               ~Eats(y, z) V Killed(y) V Food(z)
        \                       /
         \                     /    {Peanuts/z}
          \                   /
          ~Eats(y, Peanuts) V Killed(y) Eats(Anil, Peanuts)
             \                       /
              \                     /    {Anil/y}
               \                   /
               Killed(Anil)                 ~Alive(k) V ~Killed(k)
                  \                       /
                   \                     /    {Anil/k}
                    \                   /
                    ~Alive(Anil)                 Alive(Anil)
                       \                       /
                        \                     /
                         \                   /
                         {   } Hence proved.
